# Day 12 — Mini-Project Dry Run
### Dataset: Data Science Job Salaries (`ds_salaries.csv`)

This notebook combines everything from Days 1–11: Python fundamentals, pandas, NumPy, and Matplotlib.

The goal is to practice the **full workflow** you'll use in your real project starting Day 13:

1. Load & inspect
2. Clean
3. Analyze (pandas)
4. Analyze (NumPy)
5. Visualize (Matplotlib)

As always: try each cell yourself first. If you're stuck, ask and I'll show you.

**Before you start:** update the path below if your CSV isn't at `data/ds_salaries.csv`.

## 0. Setup

Import the libraries you'll need: `pandas`, `numpy`, `matplotlib.pyplot`.

Then load the CSV into a DataFrame called `df`.

In [1]:
# Import pandas, numpy, and matplotlib.pyplot with their standard aliases
# Load the CSV into a DataFrame called df
import pandas as pd
import numpy as np
import matplotlib as plt 

df = pd.read_csv("data/Data Science Jobs Salaries.csv")


In [2]:
# Test cell — just run this, don't edit
print(type(df))
df.head()


<class 'pandas.core.frame.DataFrame'>


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2021e,EN,FT,Data Science Consultant,54000,EUR,64369,DE,50,DE,L
1,2020,SE,FT,Data Scientist,60000,EUR,68428,GR,100,US,L
2,2021e,EX,FT,Head of Data Science,85000,USD,85000,RU,0,RU,M
3,2021e,EX,FT,Head of Data,230000,USD,230000,RU,50,RU,L
4,2021e,EN,FT,Machine Learning Engineer,125000,USD,125000,US,100,US,S


## 1. Inspect the Data

Before touching anything, get a feel for the dataset. Answer these using pandas methods you already know:

- How many rows and columns are there?
- What are the column dtypes?
- Are there any missing values?
- What are the unique values in `experience_level` and `company_size`?

In [3]:
# a) Print the shape of df (rows, columns)

df.shape


(245, 11)

In [39]:
# b) Print info about dtypes and non-null counts

df.dtypes


work_year             object
experience_level      object
employment_type       object
job_title             object
salary                 int64
salary_currency       object
salary_in_usd          int64
employee_residence    object
remote_ratio           int64
company_location      object
company_size          object
dtype: object

In [5]:
# c) Print the count of missing values per column
df.isnull().sum()

work_year             0
experience_level      0
employment_type       0
job_title             0
salary                0
salary_currency       0
salary_in_usd         0
employee_residence    0
remote_ratio          0
company_location      0
company_size          0
dtype: int64

In [6]:
# d) Print the unique values in 'experience_level' and 'company_size'
print(df["experience_level"].unique())
print(df["company_size"].unique()) 

['EN' 'SE' 'EX' 'MI']
['L' 'M' 'S']


In [7]:
# Test cell
assert df.shape[0] > 0, "df looks empty — check your file path"
print("Rows:", df.shape[0], "| Columns:", df.shape[1])


Rows: 245 | Columns: 11


## 2. Clean the Data

A few real-world things to handle:

1. There's often a stray index column (e.g. `Unnamed: 0`) left over from how the CSV was exported — drop it if it exists.
2. `experience_level` uses short codes: `EN`, `MI`, `SE`, `EX`. Map these to readable labels:
   `EN` → `Entry`, `MI` → `Mid`, `SE` → `Senior`, `EX` → `Executive`
3. Check for and handle any duplicate rows.

Work on a copy of df (not the original) so you don't lose the raw data — you learned why this matters back in Day 4/5.

In [8]:
# a) Make a copy of df called df_clean

# b) Create a mapping dict for experience_level codes -> full names
# Then use it to create a new column df_clean['experience_level_full']

df_clean = df.copy() 
exp_map = { "EN" : "Entry",
            "MI" : "Mid",
            "SE" : "Senior",
            "EX" : "Excutive"
}

df_clean["experience_level_full"] = df_clean["experience_level"].map(exp_map)  


In [9]:
# c) Drop the 'Unnamed: 0' column from df_clean if it exists
# Hint: use df_clean.columns to check first, or use errors='ignore' with .drop()

if "Unnamed : 0" in df_clean.columns:
    df_clean = df_clean.drop("Unnamed : 0", axis = 1)


In [ ]:
# d) Check for duplicate rows:
df_clean.duplicated().sum()

# then drop them if any exist: 
d_clean = df_clean.drop_duplicates() 


In [ ]:
# Test cell
print(df_clean['experience_level_full'].unique())

print("Duplicates remaining:", df_clean.duplicated().sum())
df_clean.head()


['Entry' 'Senior' 'Excutive' 'Mid']
Duplicates remaining: 1


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size,experience_level_full
0,2021e,EN,FT,Data Science Consultant,54000,EUR,64369,DE,50,DE,L,Entry
1,2020,SE,FT,Data Scientist,60000,EUR,68428,GR,100,US,L,Senior
2,2021e,EX,FT,Head of Data Science,85000,USD,85000,RU,0,RU,M,Excutive
3,2021e,EX,FT,Head of Data,230000,USD,230000,RU,50,RU,L,Excutive
4,2021e,EN,FT,Machine Learning Engineer,125000,USD,125000,US,100,US,S,Entry


## 3. Analyze with pandas

Now the actual analysis. Use `groupby`, `.agg()`, and pivot tables.

1. Average `salary_in_usd` by `experience_level_full`
2. A pivot table: average `salary_in_usd` by `experience_level_full` (rows) and `company_size` (columns)
3. Top 10 highest-paid `job_title`s by average `salary_in_usd` (only include titles with at least 5 people, so you're not skewed by a single high outlier)

In [27]:
# a) Average salary_in_usd grouped by experience_level_full

average = df_clean.groupby("experience_level_full")["salary_in_usd"].mean() 
average


experience_level_full
Entry        59753.462963
Excutive    226288.000000
Mid          85738.135922
Senior      128841.298701
Name: salary_in_usd, dtype: float64

In [31]:
# b) Pivot table: avg salary_in_usd, rows=experience_level_full, columns=company_size
df_clean.pivot_table( index ="experience_level_full", columns = "company_size", values = "salary_in_usd", aggfunc = "mean")

company_size,L,M,S
experience_level_full,,,
Entry,75148.000000,41063.923077,57502.000000
Excutive,239729.875000,85000.000000,243164.500000
Mid,96285.451613,83982.800000,47610.000000
Senior,134465.604651,122572.125000,120978.055556


In [14]:
# c) Top 10 highest-paid job titles by average salary_in_usd,
# only counting titles with 5+ people
# Hint: groupby job_title, use .agg() to get both mean and count in one go, then filter

job_by_salary_avg = df_clean.groupby("job_title")["salary_in_usd"].mean() 
top_ten = job_by_salary_avg.sort_values(ascending = False).head(10) 
print(top_ten)

job_title
Financial Data Analyst                450000.000000
Applied Machine Learning Scientist    423000.000000
Principal Data Engineer               392500.000000
Head of Data                          232500.000000
Principal Data Scientist              225097.800000
Director of Data Science              197751.500000
ML Engineer                           180655.333333
Machine Learning Scientist            180500.000000
Data Architect                        180000.000000
Principal Data Analyst                170000.000000
Name: salary_in_usd, dtype: float64


In [15]:
# Test cell — just run this
print("Sections above should show grouped/pivoted salary data with no errors.")


Sections above should show grouped/pivoted salary data with no errors.


## 4. Analyze with NumPy

Convert the `salary_in_usd` column to a NumPy array and practice the stats/masking you learned Days 7–9.

1. Convert `df_clean['salary_in_usd']` to a NumPy array called `salaries`
2. Compute mean, median, std, and the 90th percentile
3. Use boolean masking to count how many salaries are above the 90th percentile
4. Use `np.where()` to create a new column `salary_band` in `df_clean`: `'High'` if salary_in_usd is above the overall mean, else `'Low'`

In [42]:
print(df_clean.dtypes)


work_year                object
experience_level         object
employment_type          object
job_title                object
salary                    int64
salary_currency          object
salary_in_usd             int64
employee_residence       object
remote_ratio              int64
company_location         object
company_size             object
experience_level_full    object
dtype: object


In [52]:
# a) Convert salary_in_usd to a NumPy array called salaries
 
salaries = df_clean["salary_in_usd"].to_numpy() 
df_clean =df_clean.drop("salaries", axis = 1)

df_clean.dtypes


work_year                object
experience_level         object
employment_type          object
job_title                object
salary                    int64
salary_currency          object
salary_in_usd             int64
employee_residence       object
remote_ratio              int64
company_location         object
company_size             object
experience_level_full    object
dtype: object

In [54]:
# b) Compute mean, median, std, and 90th percentile of salaries
# Print them all with clear labels

avg = np.mean(salaries)
the_median = np.median(salaries)
std_ = np.std(salaries)
the_percentile = np.percentile(salaries, 90)

print("Mean:", avg)
print("Median:", the_median)
print("Std Dev:", std_)
print("90th Percentile:", the_percentile) 

Mean: 99868.01224489795
Median: 81000.0
Std Dev: 83811.75715411935
90th Percentile: 187966.8


In [61]:
# c) Count how many salaries are above the 90th percentile using boolean masking

above_the_percentile = salaries[ salaries > the_percentile ]

len(above_the_percentile)


25

In [62]:
# d) Use np.where() to create df_clean['salary_band']: 'High' if above mean salary, else 'Low'

df_clean["salary_band"] = np.where(salaries > avg, "high", "Low")


In [63]:
# Test cell
print(df_clean['salary_band'].value_counts())


salary_band
Low     149
high     96
Name: count, dtype: int64


## 5. Visualize with Matplotlib

Three plots:

1. **Bar chart** — average salary_in_usd by experience_level_full
2. **Histogram** — distribution of salary_in_usd (try 30 bins)
3. **Box plot** — salary_in_usd grouped by company_size

Label your axes and give each plot a title — habits worth building now.

In [ ]:
# a) Bar chart: avg salary_in_usd by experience_level_full
# Remember: plt.bar(), plt.xlabel(), plt.ylabel(), plt.title(), plt.show()

pass


In [ ]:
# b) Histogram of salary_in_usd, 30 bins

pass


In [ ]:
# c) Box plot of salary_in_usd grouped by company_size
# Hint: you can use df_clean.boxplot(column=..., by=...) or plt.boxplot() with grouped data

pass


## Wrap-up

If everything above ran without errors, you've just done a full dry run of the workflow you'll use for your real project starting **Day 13**.

Quick reflection (no code needed, just think about it):
- Which part felt shakiest — cleaning, pandas analysis, NumPy, or plotting?
- Did anything about this dataset surprise you compared to the toy examples we've used so far?

Bring your answers when we start Day 13 — it'll help decide where we spend extra time.